# Logistic Regression: Sigmoid, Loss Function & Gradient Descent

In our previous notebook, we explored the geometric foundation of classification via the **Perceptron Trick**. While the Perceptron is a great educational tool, it possesses critical limitations that make it unsuitable for production tasks.

In these notes, we will study how Logistic Regression resolves the Perceptron's flaws by introducing:
1. **The Sigmoid Function:** Moving from hard boundary steps to smooth, probabilistic classification with a "Push-Pull" strategy.
2. **Binary Cross Entropy (Log Loss):** Formulating an optimization metric using **Maximum Likelihood Estimation (MLE)**.
3. **Gradient Descent:** Deriving the matrix-form weight update equations and coding a complete NumPy implementation of Logistic Regression from scratch.

## 1. From Perceptron to Sigmoid: The Push-Pull Strategy

### The Flaw of the Perceptron
The Perceptron algorithm relies on a binary step function. If a training point is correctly classified, the model makes **zero updates** to its weights. 
- **The Problem:** Once the model finds *any* line that separates the two classes, it immediately stops learning. 
- **The Result:** The resulting boundary line is often positioned extremely close to one of the classes, leading to poor generalization (low margin) and high sensitivity to noise.

### The New Strategy: Push and Pull
To find an optimal, maximum-margin boundary, we need an algorithm that continuously refines the boundary even when points are classified correctly. This is achieved through a **Push and Pull** strategy:
1. **Misclassified Points:** "Pull" the decision boundary closer to correct the error.
2. **Correctly Classified Points:** "Push" the decision boundary away to maximize the margin of safety.
3. **Distance Sensitivity:** The intensity of the push or pull should depend on the point's distance from the line.
   - Points **close** to the boundary should exert a **strong** influence.
   - Points **far** from the boundary should exert a **weak** (near-zero) influence.

### Introducing the Sigmoid Function
To implement this distance-sensitive strategy, we replace the Perceptron's hard step function with a smooth, S-shaped curve: the **Sigmoid Function** (also called the logistic function).

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Where $z = W^T X = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_d x_d$.

#### Key Mathematical Properties:
- **Output Range:** Maps any real-valued input $z \in (-\infty, \infty)$ to a value in the interval $(0, 1)$.
- **Probabilistic Interpretation:** We can interpret the output $\hat{y} = \sigma(z)$ as the probability that the input $X$ belongs to the positive class ($y = 1$).
  - $P(y = 1 | X) = \hat{y}$
  - $P(y = 0 | X) = 1 - \hat{y}$
- **Decision Boundary:** An output of exactly $0.5$ occurs when $z = 0$, representing the linear boundary.
- **Continuous Gradient:** Unlike the step function, the Sigmoid function is differentiable everywhere. The prediction $\hat{y}$ is never exactly $0$ or $1$ (it only approaches them asymptotically). Consequently, the error term $(y_i - \hat{y}_i)$ is **never exactly zero**, meaning *every* data point will contribute a gradient update, constantly pushing or pulling the line to achieve an optimal margin.

Let's visualize the Sigmoid function first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

# Define Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.linspace(-10, 10, 200)
sigmoid_values = sigmoid(z_values)

plt.figure(figsize=(10, 5))
plt.plot(z_values, sigmoid_values, color='#9b59b6', linewidth=3, label=r'$\sigma(z) = \frac{1}{1 + e^{-z}}$')
plt.axvline(0, color='gray', linestyle='--', alpha=0.7)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7)
plt.title('The Sigmoid Function', fontsize=14, fontweight='bold')
plt.xlabel('$z = W^T X$', fontsize=12)
plt.ylabel('Predicted Probability ($\hat{y}$)', fontsize=12)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 2. The Loss Function: Maximum Likelihood Estimation (MLE) & Binary Cross Entropy

To train Logistic Regression systematically, we need an **objective metric** (a Loss Function) that quantifies classification errors. 
> **Why not use MSE (Mean Squared Error)?**
> If we combine the non-linear Sigmoid function with Mean Squared Error, the resulting loss landscape becomes **non-convex**. It will contain numerous local minima and saddle points, causing Gradient Descent to get stuck. Instead, we derive a convex loss function using probability theory.

### Step 1: Formulating Individual Probability
For a binary outcome $y_i \in \{0, 1\}$, let the model's prediction be $\hat{y}_i = \sigma(W^T X_i)$. The probability of observing label $y_i$ can be written in a single consolidated equation:

$$P(y_i | X_i; W) = \hat{y}_i^{y_i} (1 - \hat{y}_i)^{1 - y_i}$$

Let's verify this formula:
- If $y_i = 1 \implies P(y_i | X_i) = \hat{y}_i^1 (1 - \hat{y}_i)^0 = \hat{y}_i$ (Correct)
- If $y_i = 0 \implies P(y_i | X_i) = \hat{y}_i^0 (1 - \hat{y}_i)^1 = 1 - \hat{y}_i$ (Correct)

### Step 2: Maximum Likelihood Estimation (MLE)
Assuming that all $n$ data points are independent and identically distributed (i.i.d.), the joint probability of observing the entire training labels dataset is the product of individual probabilities:

$$L(W) = \prod_{i=1}^{n} \hat{y}_i^{y_i} (1 - \hat{y}_i)^{1 - y_i}$$

Our objective in MLE is to find the parameter vector $W$ that **maximizes** this likelihood function $L(W)$.
- **The Computational Limitation:** Multiplying many small probabilities (values between 0 and 1) results in extremely tiny floats, leading to **floating-point underflow** (rounding to absolute 0).

### Step 3: Converting to Binary Cross Entropy (Log Loss)
To resolve underflow and simplify optimization, we apply the **natural logarithm** to the likelihood function. Since $\log(a \cdot b) = \log(a) + \log(b)$, the product becomes a summation:

$$\log L(W) = \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

Since we want a **minimization** objective for standard Gradient Descent, we negate this log-likelihood and divide by the number of samples $n$ to compute the average loss. This yields the final **Binary Cross Entropy Loss (Log Loss)** function:

$$L(W) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

#### Intuitive Penalty Analysis:
- If actual $y_i = 1$: Loss is $-\log(\hat{y}_i)$. As prediction $\hat{y}_i \to 1$, loss $	o 0$. As $\hat{y}_i \to 0$, loss $	o \infty$.
- If actual $y_i = 0$: Loss is $-\log(1 - \hat{y}_i)$. As prediction $\hat{y}_i \to 0$, loss $	o 0$. As $\hat{y}_i \to 1$, loss $	o \infty$.
This strictly penalizes confident, incorrect predictions!

Let's visualize this log loss behavior.

In [ ]:
# Visualize Log Loss penalty for actual y = 1 and y = 0
y_hat_vals = np.linspace(0.001, 0.999, 100)
loss_y1 = -np.log(y_hat_vals)
loss_y0 = -np.log(1 - y_hat_vals)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(y_hat_vals, loss_y1, color='#2ecc71', linewidth=3)
plt.title('Log Loss when Actual $y = 1$', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Probability ($\hat{y}$)')
plt.ylabel('Loss Value')

plt.subplot(1, 2, 2)
plt.plot(y_hat_vals, loss_y0, color='#3498db', linewidth=3)
plt.title('Log Loss when Actual $y = 0$', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Probability ($\hat{y}$)')
plt.ylabel('Loss Value')

plt.tight_layout()
plt.show()

## 3. Gradient Descent: Derivation and Matrix Form

Since the Binary Cross Entropy loss function does not have a closed-form analytical solution (we cannot solve for $W$ directly like OLS in Linear Regression), we must use **Gradient Descent** to find the optimal weights iteratively.

### Matrix Formulations
For a dataset of $n$ samples and $d$ features:
- **Input Data Matrix ($X$):** Dimension $n \times (d + 1)$
  $$X = \begin{bmatrix} 1 & x_{11} & x_{12} & \dots & x_{1d} \\ 1 & x_{21} & x_{22} & \dots & x_{2d} \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & x_{n1} & x_{n2} & \dots & x_{nd} \end{bmatrix}$$
- **Weight Vector ($W$):** Dimension $(d + 1) \times 1$
  $$W = \begin{bmatrix} w_0 \\ w_1 \\ \vdots \\ w_d \end{bmatrix}$$
- **Actual Labels Vector ($Y$):** Dimension $n \times 1$
  $$Y = \begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{bmatrix}$$
- **Prediction Vector ($\hat{Y}$):** Dimension $n \times 1$
  $$\hat{Y} = \sigma(X W) = \begin{bmatrix} \sigma(W^T X_1) \\ \sigma(W^T X_2) \\ \vdots \\ \sigma(W^T X_n) \end{bmatrix}$$

### Step-by-Step Gradient Derivation
Let's compute the partial derivative of the loss function with respect to weight $w_j$.
Recall the loss function:
$$L(W) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

Applying the chain rule:
$$\frac{\partial L}{\partial w_j} = \sum_{i=1}^n \frac{\partial L}{\partial \hat{y}_i} \cdot \frac{\partial \hat{y}_i}{\partial z_i} \cdot \frac{\partial z_i}{\partial w_j}$$

Where $z_i = W^T X_i = \sum_{k=0}^d w_k x_{ik}$.

1. **Derivative of Loss w.r.t. Prediction ($\hat{y}_i$):**
   $$\frac{\partial L}{\partial \hat{y}_i} = -\frac{1}{n} \left( \frac{y_i}{\hat{y}_i} - \frac{1 - y_i}{1 - \hat{y}_i} \right) = -\frac{1}{n} \left( \frac{y_i(1 - \hat{y}_i) - (1 - y_i)\hat{y}_i}{\hat{y}_i(1 - \hat{y}_i)} \right) = -\frac{1}{n} \left( \frac{y_i - \hat{y}_i}{\hat{y}_i(1 - \hat{y}_i)} \right)$$

2. **Derivative of Sigmoid w.r.t. Input ($z_i$):**
   Since $\hat{y}_i = \sigma(z_i)$, we use the identity $\sigma'(z) = \sigma(z)(1 - \sigma(z))$:
   $$\frac{\partial \hat{y}_i}{\partial z_i} = \hat{y}_i (1 - \hat{y}_i)$$

3. **Derivative of Input $z_i$ w.r.t. Weight $w_j$:**
   $$\frac{\partial z_i}{\partial w_j} = x_{ij}$$

Now multiply the terms together:
$$\frac{\partial L}{\partial w_j} = \sum_{i=1}^n \left[ -\frac{1}{n} \left( \frac{y_i - \hat{y}_i}{\hat{y}_i(1 - \hat{y}_i)} \right) \cdot \hat{y}_i (1 - \hat{y}_i) \cdot x_{ij} \right] = \frac{1}{n} \sum_{i=1}^n (\hat{y}_i - y_i) x_{ij}$$

### Matrix Representation of the Gradient
Writing the partial derivatives for all weights $w_j$ in vector form:

$$\frac{\partial L}{\partial W} = \frac{1}{n} X^T (\hat{Y} - Y)$$

#### Weight Update Rule
Using the learning rate $\eta$, the weights are updated iteratively:

$$W_{\text{new}} = W_{\text{old}} - \eta \cdot \frac{\partial L}{\partial W} \implies W_{\text{new}} = W_{\text{old}} - \frac{\eta}{n} X^T (\hat{Y} - Y)$$

This update rule is incredibly clean and vectorizable, which allows libraries like NumPy to compute updates extremely fast.

## 4. NumPy Implementation of Logistic Regression

Now, let's write a complete Object-Oriented implementation of Logistic Regression using NumPy. We will implement batch Gradient Descent.

In [ ]:
class GDLogisticRegression:
    def __init__(self, lr=0.1, epochs=10000):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.loss_history = []
        
    def _sigmoid(self, z):
        # Clip values to avoid overflow/underflow warnings
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def _compute_loss(self, y_true, y_pred):
        # Add epsilon to prevent log(0)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Prepend column of 1s to X for the intercept (bias) term
        X_bias = np.insert(X, 0, 1, axis=1)
        
        # Initialize weights with 1s
        self.weights = np.ones(X_bias.shape[1])
        
        for epoch in range(self.epochs):
            # 1. Compute dot products
            z = np.dot(X_bias, self.weights)
            
            # 2. Make probabilistic predictions
            y_pred = self._sigmoid(z)
            
            # 3. Calculate and record the Log Loss
            loss = self._compute_loss(y, y_pred)
            self.loss_history.append(loss)
            
            # 4. Calculate gradient: (1/n) * X_bias^T * (y_pred - y)
            gradient = np.dot(X_bias.T, (y_pred - y)) / n_samples
            
            # 5. Update weights
            self.weights -= self.lr * gradient
            
            # Print loss updates occasionally
            if epoch % (self.epochs // 10) == 0:
                print(f"Epoch {epoch:5d} | Log Loss: {loss:.6f}")
                
        print(f"Final Epoch | Log Loss: {self.loss_history[-1]:.6f}")
        
    def predict_proba(self, X):
        X_bias = np.insert(X, 0, 1, axis=1)
        return self._sigmoid(np.dot(X_bias, self.weights))
    
    def predict(self, X, threshold=0.5):
        probas = self.predict_proba(X)
        return (probas >= threshold).astype(int)
    
    @property
    def intercept_(self):
        return self.weights[0]
        
    @property
    def coef_(self):
        return self.weights[1:]

print("Custom Logistic Regression class defined!")

## 5. Dataset Generation & Model Training

Let's generate a synthetic classification dataset and train both our custom model and Scikit-Learn's model to compare.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Generate synthetic binary dataset
X_raw, y = make_classification(
    n_samples=200, 
    n_features=2, 
    n_informative=2,
    n_redundant=0, 
    n_clusters_per_class=1, 
    class_sep=1.5, 
    random_state=42
)

# Standardize features (highly recommended for gradient descent)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# Fit custom model
print("Training Custom GDLogisticRegression model:")
custom_model = GDLogisticRegression(lr=0.5, epochs=5000)
custom_model.fit(X, y)

# Fit Scikit-Learn's model
print("\nTraining Scikit-Learn Logistic Regression model:")
lr_sklearn = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)
lr_sklearn.fit(X, y)

print("\nModels trained successfully!")

## 6. Verification and Visualizations

Let's verify:
1. That our Log Loss decreased monotonically over epochs.
2. How the decision boundaries compare.
3. How close the weight values are.

In [ ]:
# Plot Log Loss history
plt.figure(figsize=(10, 4))
plt.plot(custom_model.loss_history, color='#e74c3c', linewidth=2.5)
plt.title('Log Loss Convergence Curve (Gradient Descent)', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Binary Cross Entropy Loss', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Compare decision boundaries
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
x_plot = np.linspace(x_min, x_max, 100)

# Custom model weights
w0_c = custom_model.intercept_
w1_c, w2_c = custom_model.coef_[0], custom_model.coef_[1]

# Sklearn model weights
w0_sk = lr_sklearn.intercept_[0]
w1_sk, w2_sk = lr_sklearn.coef_[0][0], lr_sklearn.coef_[0][1]

plt.figure(figsize=(14, 6))

# Plot 1: Custom Model
plt.subplot(1, 2, 1)
plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=60, label='Class 0')
plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#2ecc71', edgecolor='k', s=60, label='Class 1')
y_plot_c = -(w1_c / w2_c) * x_plot - (w0_c / w2_c)
plt.plot(x_plot, y_plot_c, color='#e74c3c', linewidth=3, label='Custom GD')
plt.title('Custom NumPy Gradient Descent Boundary', fontsize=12, fontweight='bold')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.xlim(x_min, x_max)
plt.ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
plt.legend(frameon=True, facecolor='white')

# Plot 2: Sklearn Model
plt.subplot(1, 2, 2)
plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=60, label='Class 0')
plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#2ecc71', edgecolor='k', s=60, label='Class 1')
y_plot_sk = -(w1_sk / w2_sk) * x_plot - (w0_sk / w2_sk)
plt.plot(x_plot, y_plot_sk, color='#9b59b6', linewidth=3, label='Scikit-Learn')
plt.title('Scikit-Learn L-BFGS Boundary', fontsize=12, fontweight='bold')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.xlim(x_min, x_max)
plt.ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
plt.legend(frameon=True, facecolor='white')

plt.tight_layout()
plt.show()

# Print coefficients side-by-side
print("="*60)
print(f"{'Parameter':<15} | {'Custom NumPy (GD)':<20} | {'Scikit-Learn (L-BFGS)':<20}")
print("="*60)
print(f"{'Intercept (w0)':<15} | {w0_c:<20.6f} | {w0_sk:<20.6f}")
print(f"{'Feature 1 (w1)':<15} | {w1_c:<20.6f} | {w1_sk:<20.6f}")
print(f"{'Feature 2 (w2)':<15} | {w2_c:<20.6f} | {w2_sk:<20.6f}")
print("="*60)

## Summary Checklist for Logistic Regression Mathematics & Training

1. **Why Sigmoid?** Replacing the step function maps values to probabilities $(0, 1)$ and yields a continuous gradient, solving the Perceptron stopping issue.
2. **Log Loss (Binary Cross Entropy) over MSE:** Using Log Loss guarantees a convex optimization surface, meaning Gradient Descent will converge to the global minimum.
3. **Use MLE for Formulation:** Maximum Likelihood Estimation maximizes the probability of getting the correct predictions, and taking the negative logarithm leads to the minimization of Log Loss:
   $$L(W) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$
4. **Vectorized Gradients:** Compute gradients using the matrix dot product $\frac{1}{n} X^T (\hat{Y} - Y)$ for high-performance training.
5. **Feature Scaling Requirement:** Always scale features (e.g. standard scaling) when using Gradient Descent to ensure faster and smoother convergence.